In [27]:
#Install dependencies
!pip install -q \
transformers==4.52.4 \
datasets==3.6.0 \
peft==0.15.2 \
accelerate==1.7.0 \
trl==0.18.1 \
evaluate \
sentencepiece \
rouge_score \
sacrebleu

In [28]:
#verify dependencies
import transformers
import peft
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("PyTorch:", torch.__version__)

Transformers: 4.52.4
PEFT: 0.15.2
Accelerate: 1.7.0
Datasets: 3.6.0
PyTorch: 2.11.0+cu128


In [37]:
#Load Model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(model_name)

print("✅ Model Loaded!")

Loading tokenizer...
Loading model...
✅ Model Loaded!


In [41]:
#load dataset
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [42]:
#Train validation split
dataset = dataset["train"].shuffle(seed=42)

train_data = dataset.select(range(10000))
val_data = dataset.select(range(10000,12000))

print(len(train_data))
print(len(val_data))

10000
2000


In [43]:
#Processing
MAX_LENGTH = 256

def preprocess(example):

    if example["input"].strip():

        text = (
            f"Instruction: {example['instruction']}\n"
            f"Input: {example['input']}\n"
            f"Answer: {example['output']}"
        )

    else:

        text = (
            f"Instruction: {example['instruction']}\n"
            f"Answer: {example['output']}"
        )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

print("✅ Preprocessing Ready")

✅ Preprocessing Ready


In [44]:
#tokenize dataset
tokenized_train = train_data.map(
    preprocess,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    preprocess,
    remove_columns=val_data.column_names
)

print("✅ Tokenization Complete")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Tokenization Complete


In [45]:
#Apply LoRA
from peft import LoraConfig,get_peft_model,TaskType

lora_config = LoraConfig(

    task_type=TaskType.CAUSAL_LM,

    inference_mode=False,

    r=16,

    lora_alpha=32,

    lora_dropout=0.1,

    target_modules=["c_attn"]

)

model = get_peft_model(model,lora_config)

model.print_trainable_parameters()

trainable params: 294,912 || all params: 82,207,488 || trainable%: 0.3587


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [46]:
#Data collected
from transformers import default_data_collator

data_collator = default_data_collator

print("✅ Data Collator Ready")

✅ Data Collator Ready


In [47]:
#Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./alpaca_lora",

    num_train_epochs=5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=2e-4,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=1,

    fp16=True,

    report_to="none",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False

)

print("✅ Training Arguments Ready")

✅ Training Arguments Ready


In [48]:
#Trainer
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    tokenizer=tokenizer,

    data_collator=data_collator

)

print("✅ Trainer Ready")

/tmp/ipykernel_12958/4034854809.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ Trainer Ready


In [49]:
#Train
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.469700,1.418761
2,1.391400,1.398188
3,1.430300,1.388340
4,1.419400,1.383654
5,1.385600,1.381488


TrainOutput(global_step=12500, training_loss=1.479321904296875, metrics={'train_runtime': 894.5604, 'train_samples_per_second': 55.893, 'train_steps_per_second': 13.973, 'total_flos': 3288858624000000.0, 'train_loss': 1.479321904296875, 'epoch': 5.0})

In [50]:
trainer.args.max_steps = -1

In [51]:
model.save_pretrained("alpaca_lora_model")
tokenizer.save_pretrained("alpaca_lora_model")

print("✅ Model Saved Successfully!")

✅ Model Saved Successfully!


In [52]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100
)

def ask(question):
    prompt = f"Instruction: {question}\nAnswer:"
    result = generator(
        prompt,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    print(result[0]["generated_text"])

Device set to use cuda:0


In [53]:
ask("Write a short motivational quote.")

Instruction: Write a short motivational quote.
Answer: "I'm glad I've learned to live in a world where my dreams are limitless."


In [54]:
model.save_pretrained("alpaca_lora_model")

tokenizer.save_pretrained("alpaca_lora_model")

print("✅ Model Saved")

✅ Model Saved


In [55]:
import torch

def ask(question):

    prompt = f"Instruction: {question}\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(

        **inputs,

        max_new_tokens=80,

        temperature=0.6,

        top_p=0.9,

        repetition_penalty=1.2,

        no_repeat_ngram_size=3,

        do_sample=True,

        eos_token_id=tokenizer.eos_token_id,

        pad_token_id=tokenizer.eos_token_id

    )

    print(tokenizer.decode(outputs[0],skip_special_tokens=True))

In [56]:
ask("Explain machine learning.")

Instruction: Explain machine learning.
Answer: Machine Learning is a mathematical concept that has been used to improve the performance of computer programs and applications by increasing efficiency in processing data, reducing errors or delays due not being fully processed properly. This approach allows for better understanding how information flows between different systems using algorithms such as RNNs, which are implemented over time with minimal overhead on each platform's processors. It provides an efficient way to predict what will


In [57]:
ask("How do I improve my communication skills?")

Instruction: How do I improve my communication skills?
Answer: Here are some ways to increase your overall effectiveness by using the following tips. 
- Use a simple, intuitive way of communicating with other people and communicate effectively - use an app like WhatsApp or Skype for example that lets you share messages directly between friends in real time; follow up on social media posts as well as video chats (such be it Facebook Messenger), making sure they’re shared regularly


In [58]:
ask("Write a motivational quote.")

Instruction: Write a motivational quote.
Answer: "I hope that you will give me the courage to help others in their journey."
